In [2]:
#Test
import dagster as dg
from dagster import op, job, In, Out, DynamicOut, DynamicOutput, multiprocess_executor
from dataclasses import dataclass
from typing import List, Tuple, Optional
from collections import defaultdict

import math
import numpy as np

# ---------------------------
# SPECS / RESULTS
# ---------------------------

@dataclass
class Node1Spec:
    n: int
    p: int

@dataclass
class Node2Spec:
    use_numpy: bool = False

@dataclass
class Node2Result:
    value: int
    is_prime: bool
   
@dataclass
class Node3Spec:
    use_numpy: bool = False
    binwidth: int = 100000
    mode: str = "per_partition"
    
from typing import Dict

@dataclass
class Node3Result:
    counts: Dict[int, int]


# ---------------------------
# Non Ops 
# ---------------------------

class X:
    def __init__(self, n: int):
        self.n = n
        
    def mult(self, x: int) -> int:
        return self.n * x

# ---------------------------
# NODES
# ---------------------------

# NODE 1: Partitionierung + Fan-Out
@op(out=DynamicOut())
def node1_partition(n: int, p: int):
    """
    Für jedes m = 1 .. p-1:
    - erzeuge die ersten n Zahlen >= 2, die kongruent m mod p sind
    - yield als eigener DynamicOutput mit mapping_key = part_{m}
    """
    
    for m in range(1, p):
        part: List[int] = []

        k = 0
        while len(part) < n:
            candidate = m + k * p
            if candidate >= 2:
                part.append(candidate)
            k += 1

        yield DynamicOutput(
            value=part,
            mapping_key=f"part_{m}",
        )
        
# NODE 2: Primzahlenprüfung
@op
def node2_prime_check(partition: List[int]) -> List[Node2Result]:
    use_numpy = False
    results = []
    for num in partition:
        if num < 2:
            results.append(Node2Result(num, False))
            continue
        prime = True
        for i in range(2, int(math.isqrt(num)) + 1):
            if num % i == 0:
                prime = False
                break
        results.append(Node2Result(num, prime))
    if use_numpy:
        results = np.array(results)

    x = X(2)
    y = x.mult(5)
    print(y)
    return results


# NODE 3: Statistik / Binning
@op
def node3_stat(partition_results: List[Node2Result]) -> Node3Result:

    binwidth = 10000
    counts = defaultdict(int)

    for r in partition_results:
        if r.is_prime:
            idx = r.value // binwidth
            counts[idx] += 1
    x = X(3)
    y = x.mult(5)
    print(y)
    return Node3Result(counts=dict(counts))


# NODE 4: Aggregation
@op(out=Out(Node3Result))
def node4_aggregate(stats_list: List[Node3Result]) -> Node3Result:
    agg = defaultdict(int)

    for stats in stats_list:
        for idx, cnt in stats.counts.items():
            agg[idx] += cnt
  
    return Node3Result(counts=dict(agg))